### 퓨샷 프롬프트

In [ ]:
# ===== 패키지 설치 (최초 1회만 실행) =====
!pip install python-dotenv
!pip install -U langchain langchain-openai langchain-teddynote

# ===== 필요한 모듈 import =====
import os
from dotenv import load_dotenv
from langchain_teddynote import logging
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate, load_prompt
from datetime import datetime

# ===== 환경변수(.env) 로드 =====
load_dotenv()  # override=False가 기본값이므로 시스템 환경변수가 우선순위를 가짐

# (선택) 정상 로드 확인
print("OpenAI 키:", os.getenv("OPENAI_API_KEY")[:8] + "...")
print("LANGSMITH 키:", os.getenv("LANGSMITH_API_KEY")[:8] + "...")
print("LangSmith 프로젝트:", os.getenv("LANGSMITH_PROJECT"))

# ===== LangSmith 추적 시작 =====
logging.langsmith("CH02-Prompt")  # 프로젝트명 입력

# ===== LLM 객체 생성 =====
llm = ChatOpenAI()

In [3]:
from langchain_core.prompts.few_shot import FewShotPromptTemplate
from langchain_core.output_parsers import StrOutputParser

examples = [
    {
    "question": "스티브잡스와 아인슈타인 중 누가 더 오래 살았나요?",
    "answer": """이 질문에 추가 질문이 필요한가요: 예.
                추가 질문 : 스티브 잡스는 몇 살에 사망했나요?
                중간 답변 : 스티브 잡스는 56세에 사망했습니다.
                추가 질문: 아인슈타인은 몇 살에 사망했나요?
                중간 답변 아인슈타인은 76세에 사망했습니다.
                최종 답변은 : 아인슈타인
                """,
    },
]

In [4]:
example_prompt = PromptTemplate.from_template(
    "Question:\n{question}\nAnswer:\n{answer}"
)

print(example_prompt.format(**examples[0]))

Question:
스티브잡스와 아인슈타인 중 누가 더 오래 살았나요?
Answer:
이 질문에 추가 질문이 필요한가요: 예.
                추가 질문 : 스티브 잡스는 몇 살에 사망했나요?
                중간 답변 : 스티브 잡스는 56세에 사망했습니다.
                추가 질문: 아인슈타인은 몇 살에 사망했나요?
                중간 답변 아인슈타인은 76세에 사망했습니다.
                최종 답변은 : 아인슈타인
                


In [6]:
prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix="Question:\n{question}\nAnswer:",
    input_variables=["question"],
)

question = "Google이 창립된 연도에 Bill Gates의 나이는 몇 살인가요?"
final_prompt = prompt.format(question=question)
print(final_prompt)

Question:
스티브잡스와 아인슈타인 중 누가 더 오래 살았나요?
Answer:
이 질문에 추가 질문이 필요한가요: 예.
                추가 질문 : 스티브 잡스는 몇 살에 사망했나요?
                중간 답변 : 스티브 잡스는 56세에 사망했습니다.
                추가 질문: 아인슈타인은 몇 살에 사망했나요?
                중간 답변 아인슈타인은 76세에 사망했습니다.
                최종 답변은 : 아인슈타인
                

Question:
Google이 창립된 연도에 Bill Gates의 나이는 몇 살인가요?
Answer:


In [7]:
from langchain_openai import ChatOpenAI
from langchain_teddynote.messages import stream_response

llm = ChatOpenAI() # 객체 생성

answer = llm.stream(final_prompt)
stream_response(answer) # 결과 출력

이 질문에 추가 정보가 필요합니다. 구글이 창립된 연도와 Bill Gates의 출생 연도를 알아야 합니다.

In [8]:
prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix="Question:\n{question}\nAnswer:",
    input_variables=["question"],
)

chain = prompt | llm | StrOutputParser() # 체인 생성

answer = chain.stream( # 결과출력
    {"question": "Google이 창립된 연도에 Bill Gates의 나이는 몇 살인가요?"}
)
stream_response(answer)

이 질문에 추가 질문이 필요한가요? 예
        추가 질문: 구글이 창립된 연도는 언제인가요?
        중간 답변: 구글이 창립된 연도는 1998년입니다.
        추가 질문: 빌 게이츠는 1998년에 몇 살이었나요?
        중간 답변: 빌 게이츠는 43세였습니다.
        최종 답변: 빌 게이츠는 1998년에 43세이었습니다.